# 02 · Read the AMFI reports and check them
Turns 91 monthly Excel reports and AMFI's SIP table into two clean tables, and checks every number before we use it.

### Find the project folder

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_AMFI = ROOT / "data" / "raw" / "amfi"
OUT = ROOT / "data"
pd.set_option("display.width", 160)

### Look at one report first
Each report lists every scheme category with its money in, money out, **net inflow** and **AUM** (assets under management).
Section headers such as "II Equity Schemes" sit above their categories, and a "Sub Total" row closes each section.

In [2]:
data = pd.read_excel(RAW_AMFI / "amoct2026repo.xls", sheet_name=0, header=None)   # header=None: we find the header ourselves
data.iloc[22:40, :8]

,0,1,2,3,4,5,6,7
22,NaN,Sub Total - I,343,9931249,1415109.836998,1301671.946031,113437.890967,1881341.373218
23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,II,Equity Schemes,NaN,NaN,NaN,NaN,NaN,NaN
25,i,Multi Cap Fund,33,12155011,5023.606524,2045.741866,2977.864658,243406.367918
26,ii,Large Cap Fund,35,16929204,4338.740079,3306.348338,1032.39174,393541.427684
27,iii,Large & Mid Cap Fund,36,14602910,10996.925923,2512.691696,8484.234227,355311.154632
28,iv,Mid Cap Fund,34,26983457,9001.393705,4108.769783,4892.623922,503433.672946
29,v,Small Cap Fund,36,30704519,10254.718424,3847.221502,6407.496923,440874.571423
30,vi,Flexi Cap Fund,47,25180630,9958.288093,4978.334186,4979.953907,591363.715207
31,vii,Dividend Yield Fund,12,1179983,302.337686,395.16048,-92.822794,30095.484525


### Find the columns we need
Column positions differ between older and newer reports, so we find them by their header text:
the **net inflow** column and the **AUM at month end** column (not the monthly average AUM).

In [3]:
import re

def find_columns(sheet):
    for i, row in sheet.iterrows():
        text = [str(v).lower() for v in row.tolist()]
        net = [j for j, t in enumerate(text) if "net inflow" in t]
        if net:                                                    # this is the header row
            aum = [j for j, t in enumerate(text)
                   if ("net assets" in t or "aum as on" in t or "net aum" in t) and "average" not in t and "aaum" not in t]
            return i, net[0], aum[0]
    raise ValueError("header row not found")

find_columns(data)   # (header row, net inflow column, AUM column)

(2, 6, 7)

### Read one report into rows
We walk down the sheet and keep track of where we are:
- **part**: A open-ended, B close-ended, C interval schemes
- **section**: e.g. "Equity Schemes" (newer reports) or "Growth/Equity Oriented Schemes" (older reports)
- each row is a **category**, a **subtotal**, a part **total** or the **grand total**

In [4]:
def number(v):
    try:
        x = float(v)
    except (TypeError, ValueError):
        return None
    return None if pd.isna(x) else x              # blank cells come back as NaN: treat them as "no number"

def read_report(path, month):
    sheet = pd.read_excel(path, sheet_name=0, header=None)
    header, net_col, aum_col = find_columns(sheet)
    rows, part, section = [], None, None
    for i in range(header + 1, len(sheet)):
        cells = sheet.iloc[i].tolist()
        words = [str(c).strip() for c in cells[:net_col] if isinstance(c, str) and str(c).strip()]   # text cells left of the numbers
        if not words:
            continue
        label = words[-1]                                    # the scheme or section name
        tag = words[0] if len(words) > 1 else ""             # "A", "II", "iv", "12" ...
        low = " ".join(words).lower()
        net, aum = number(cells[net_col]), number(cells[aum_col])
        if low.startswith("grand total"):
            kind = "grand_total"
        elif low.startswith("total"):
            kind = "part_total"
        elif low.startswith("sub total"):
            kind = "subtotal"
        elif net is None:                                    # a heading row: part or section
            if tag in ("A", "B", "C") or "ended" in low or "interval" in low:
                part = {"A": "open", "B": "close", "C": "interval"}.get(tag) or (
                    "open" if "open" in low else "close" if "close" in low else "interval")
                section = None
            elif re.fullmatch(r"[IVX]+", tag):
                section = label
            continue
        else:
            kind = "category"
        if "fund of funds scheme (domestic)" in low:         # a memo line below the grand total, not part of it
            continue
        if part == "interval" and kind == "category" and section is None:
            section = "Interval Schemes"
        rows.append({"month": month, "part": part, "section": section, "kind": kind, "label": label,
                     "net_inflow_cr": net, "aum_cr": aum})
    return pd.DataFrame(rows)

data = read_report(RAW_AMFI / "amoct2026repo.xls", "2026-10")
data[data["section"].fillna("").str.contains("Equity") & (data["part"] == "open")]

,month,part,section,kind,label,net_inflow_cr,aum_cr
18,2026-10,open,Equity Schemes,category,Multi Cap Fund,2977.864658,2.434064e+05
19,2026-10,open,Equity Schemes,category,Large Cap Fund,1032.391740,3.935414e+05
20,2026-10,open,Equity Schemes,category,Large & Mid Cap Fund,8484.234227,3.553112e+05
21,2026-10,open,Equity Schemes,category,Mid Cap Fund,4892.623922,5.034337e+05
22,2026-10,open,Equity Schemes,category,Small Cap Fund,6407.496923,4.408746e+05
23,2026-10,open,Equity Schemes,category,Flexi Cap Fund,4979.953907,5.913637e+05
24,2026-10,open,Equity Schemes,category,Dividend Yield Fund,-92.822794,3.009548e+04
25,2026-10,open,Equity Schemes,category,Value Fund,556.223513,1.367025e+05
26,2026-10,open,Equity Schemes,category,Contra Fund,1694.125472,7.247170e+04
27,2026-10,open,Equity Schemes,category,Focused Fund,925.350098,1.803691e+05


### Read all 91 reports

In [5]:
files = pd.read_csv(ROOT / "data" / "raw" / "amfi_files.csv")
files = files[files["kind"] == "xls"]                               # only months with an Excel report
data = pd.concat([read_report(RAW_AMFI / f, m) for f, m in zip(files["file"], files["month"])], ignore_index=True)
print(data["month"].nunique(), "months,", len(data), "rows")
data["kind"].value_counts()

91 months, 5484 rows


kind
category       4474
subtotal        646
part_total      273
grand_total      91
Name: count, dtype: int64

### Check 1: every section adds up
The categories inside each section must add up to that section's subtotal (to within Rs 1 crore of rounding).

In [6]:
data["block"] = (data["kind"] != "category").cumsum()               # each subtotal closes the categories just above it
cats = data[data["kind"] == "category"].groupby("block")["net_inflow_cr"].sum()
subs = data[data["kind"] == "subtotal"].set_index("block")
check1 = subs.assign(categories=cats.reindex(subs.index - 1).values)   # categories sit in the block before their subtotal
check1["gap"] = (check1["net_inflow_cr"] - check1["categories"]).abs()
print("sections checked:", len(check1), "· failing:", (check1["gap"] > 1).sum())
check1[check1["gap"] > 1][["month", "section", "net_inflow_cr", "categories", "gap"]]

sections checked: 646 · failing: 0


,month,section,net_inflow_cr,categories,gap
block,,,,,


### Check 2: the parts add up to the grand total

In [7]:
parts = data[data["kind"] == "part_total"].groupby("month")["net_inflow_cr"].sum()
grand = data[data["kind"] == "grand_total"].set_index("month")["net_inflow_cr"]
check2 = pd.DataFrame({"parts": parts, "grand_total": grand})
check2["gap"] = (check2["parts"] - check2["grand_total"]).abs()
print("months checked:", len(check2), "· failing:", (check2["gap"] > 1).sum())
check2[check2["gap"] > 1]

months checked: 91 · failing: 0


,parts,grand_total,gap
month,,,


### Name each section the same way across years
Older reports say "Growth/Equity Oriented Schemes" and newer ones say "Equity Schemes". We map every section to one group.
Index funds, ETFs and overseas fund of funds are grouped as **passive & other** (older reports list them together as "Other Schemes").

In [8]:
def group(part, section):
    s = (section or "").lower()
    if part != "open":
        return "Closed-ended & interval"
    if "equity" in s:
        return "Equity (active)"
    if "debt" in s or "income" in s:
        return "Debt"
    if "hybrid" in s:
        return "Hybrid"
    if "solution" in s or "life cycle" in s:
        return "Solution-oriented & life cycle"
    return "Passive & other"          # index funds, ETFs, overseas FoF, "Other Schemes"

data["group"] = [group(p, s) for p, s in zip(data["part"], data["section"])]
data[data["kind"] == "subtotal"].groupby(["group"])["section"].unique()

group
Closed-ended & interval           [Income/Debt Oriented Schemes, Growth/Equity O...
Debt                                   [Income/Debt Oriented Schemes, Debt Schemes]
Equity (active)                    [Growth/Equity Oriented Schemes, Equity Schemes]
Hybrid                                                             [Hybrid Schemes]
Passive & other                   [Other Schemes, Index Funds, Exchange Traded F...
Solution-oriented & life cycle    [Solution Oriented Schemes, Life Cycle Funds, ...
Name: section, dtype: object

### Name equity categories the same way across years
Category names changed a little over the years (Flexi Cap started in late 2020; "Value Fund/Contra Fund" was later split).

In [9]:
def equity_category(label):
    l = label.lower()
    if "large & mid" in l or "large and mid" in l: return "Large & mid cap"
    if "large cap" in l: return "Large cap"
    if "mid cap" in l: return "Mid cap"
    if "small cap" in l: return "Small cap"
    if "flexi" in l: return "Flexi cap"
    if "multi cap" in l: return "Multi cap"
    if "sector" in l or "thematic" in l: return "Sectoral & thematic"
    if "elss" in l: return "ELSS (tax saver)"
    return "Value, contra, focused & dividend yield"

eq = data[(data["group"] == "Equity (active)") & (data["kind"] == "category")].copy()
eq["category"] = eq["label"].map(equity_category)
eq.groupby("category")["label"].unique()

category
ELSS (tax saver)                                                [ELSS, ELSS- Tax Saver Fund]
Flexi cap                                                                   [Flexi Cap Fund]
Large & mid cap                                                       [Large & Mid Cap Fund]
Large cap                                                                   [Large Cap Fund]
Mid cap                                                                       [Mid Cap Fund]
Multi cap                                               [Multi Cap Fund, Multi Cap Fund  $$]
Sectoral & thematic                        [Sectoral/Thematic Funds, Sectoral Fund, Thema...
Small cap                                                                   [Small Cap Fund]
Value, contra, focused & dividend yield    [Dividend Yield Fund, Value Fund/Contra Fund, ...
Name: label, dtype: object

### Build the clean tables
- `equity`: one row per month with the equity net inflow (the open-ended equity subtotal) and equity AUM
- `groups`: net inflow per month and group
- `equity_categories`: net inflow per month and equity category

In [10]:
equity = (data[(data["group"] == "Equity (active)") & (data["kind"] == "subtotal")]
          .groupby("month")[["net_inflow_cr", "aum_cr"]].sum()
          .rename(columns={"net_inflow_cr": "equity_inflow_cr", "aum_cr": "equity_aum_cr"}))
groups = data[data["kind"] == "subtotal"].groupby(["month", "group"])["net_inflow_cr"].sum().reset_index()
total = data[data["kind"] == "grand_total"].set_index("month")[["net_inflow_cr", "aum_cr"]].rename(
    columns={"net_inflow_cr": "total_inflow_cr", "aum_cr": "total_aum_cr"})
equity_categories = eq.groupby(["month", "category"])["net_inflow_cr"].sum().reset_index()
print(len(equity), "months of equity flows")
equity.tail()

91 months of equity flows


,equity_inflow_cr,equity_aum_cr
month,,
2026-06,28973.413287,3.733731e+06
2026-07,24697.385391,3.836192e+06
2026-08,29328.617931,3.921147e+06
2026-09,38306.552049,3.754275e+06
2026-10,32993.637260,3.714550e+06


### Check 3: compare with figures AMFI and the press reported
Three months picked before looking at our results: the October 2026 AMFI Monthly Note, and two news reports of AMFI data.

In [11]:
reported = {"2026-10": 32993.64, "2026-01": 24028.59, "2025-01": 39687.78}   # Rs crore, equity net inflow
for m, value in reported.items():
    ours = equity.loc[m, "equity_inflow_cr"]
    print(m, "reported", value, "· ours", round(ours, 2), "· match" if abs(ours - value) < 1 else "· MISMATCH")

2026-10 reported 32993.64 · ours 32993.64 · match
2026-01 reported 24028.59 · ours 24028.59 · match
2025-01 reported 39687.78 · ours 39687.78 · match


### Read the SIP table
AMFI's page shows months as rows and financial years (April to March) as columns. We turn it into one row per month.

In [12]:
tables = pd.read_html(RAW_AMFI / "sip_page.html")
sip_wide = tables[0]
sip_wide.columns = [c[-1] if isinstance(c, tuple) else c for c in sip_wide.columns]   # keep the bottom header row
sip_wide = sip_wide.rename(columns={sip_wide.columns[0]: "month_name"})
sip_wide = sip_wide[sip_wide["month_name"] != "Total during FY"]
sip_wide.head(3)

,month_name,FY 2026-27,FY 2025-26,FY 2024-25,FY 2023-24,FY 2022-23,FY 2021-22,FY 2020-21,FY 2019-20,FY 2018-19,FY 2017-18,FY 2016-17
1,March,NaN,32087,25926,19271,14276,12328,9182,8641,8055,7119,4335
2,February,NaN,29845,25999,19187,13686,11438,7528,8513,8095,6425,4050
3,January,NaN,31002,26400,18838,13856,11517,8023,8532,8064,6644,4095


In [13]:
MONTHS = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
rows = []
for col in sip_wide.columns[1:]:
    start = int(re.search(r"(\d{4})", col).group(1))            # "FY 2025-26" starts in April 2025
    for name, value in zip(sip_wide["month_name"], sip_wide[col]):
        if pd.isna(value):
            continue
        m = MONTHS[str(name).strip().lower()[:3]]
        year = start if m >= 4 else start + 1                     # January to March belong to the next calendar year
        rows.append({"month": f"{year}-{m:02d}", "sip_cr": float(value)})
sip = pd.DataFrame(rows).sort_values("month").set_index("month")
print(len(sip), "months, from", sip.index.min(), "to", sip.index.max())
sip.tail()

127 months, from 2016-04 to 2026-10


,sip_cr
month,
2026-06,31781.0
2026-07,31961.0
2026-08,32297.0
2026-09,32490.0
2026-10,32160.0


### Check 4: SIP figures
October 2026 must be Rs 32,160 crore (AMFI Monthly Note). AMFI's table shows the same value, Rs 31,002 crore, for
December 2025 and January 2026. News reports of AMFI data confirm both months, so it is a real coincidence, not a copying error.

In [14]:
print("Oct 2026:", sip.loc["2026-10", "sip_cr"], "(note: 32,160)")
print("Dec 2025:", sip.loc["2025-12", "sip_cr"], "· Jan 2026:", sip.loc["2026-01", "sip_cr"], "(both reported as 31,002)")
dupes = sip[sip["sip_cr"].duplicated(keep=False)]
dupes

Oct 2026: 32160.0 (note: 32,160)
Dec 2025: 31002.0 · Jan 2026: 31002.0 (both reported as 31,002)


,sip_cr
month,
2018-06,7554.0
2018-07,7554.0
2018-10,7985.0
2018-11,7985.0
2025-12,31002.0
2026-01,31002.0


### Log every check and save the tables

In [15]:
log = pd.DataFrame([
    {"check": "categories add up to section subtotals", "checked": len(check1), "failed": int((check1["gap"] > 1).sum())},
    {"check": "parts add up to grand total", "checked": len(check2), "failed": int((check2["gap"] > 1).sum())},
    {"check": "equity inflow matches reported figures", "checked": len(reported),
     "failed": sum(abs(equity.loc[m, "equity_inflow_cr"] - v) >= 1 for m, v in reported.items())},
    {"check": "SIP October 2026 matches the Monthly Note", "checked": 1, "failed": int(sip.loc["2026-10", "sip_cr"] != 32160)},
])
log.to_csv(OUT / "extraction_log.csv", index=False)
data.drop(columns="block").to_parquet(OUT / "report_rows.parquet", index=False)
equity.to_parquet(OUT / "equity.parquet")
groups.to_parquet(OUT / "groups.parquet", index=False)
equity_categories.to_parquet(OUT / "equity_categories.parquet", index=False)
sip.to_parquet(OUT / "sip.parquet")
log

,check,checked,failed
0,categories add up to section subtotals,646,0
1,parts add up to grand total,91,0
2,equity inflow matches reported figures,3,0
3,SIP October 2026 matches the Monthly Note,1,0
